## Đọc dữ liệu

In [1]:
# Nạp thư viện os (Operating System) — dùng để tương tác với hệ điều hành (ví dụ: quản lý tệp tin, nối đường dẫn thư mục, kiểm tra tệp tồn tại).
import os
# Nạp thư viện numpy và đặt tên viết tắt là np — dùng để xử lý tính toán đại số tuyến tính, mảng nhiều chiều (array) và các phép toán số học tốc độ cao.
import numpy as np
# Nạp thư viện pandas và đặt tên viết tắt là pd — dùng để làm việc với dữ liệu dạng bảng (DataFrame), hỗ trợ đọc/ghi tệp CSV, Excel và tiền xử lý dữ liệu
import pandas as pd

# Ghép đường dẫn giữa thư mục 'path' (từ kagglehub) và tên tệp CSV
file_path = os.path.join(
    "..", "..", "data", "raw", "Diamonds Prices2022.csv"
)

# Đọc tệp CSV vào biến 'diamonds' (DataFrame) với dấu phân cách các cột là dấu phẩy (,)
diamonds = pd.read_csv(file_path, sep=',')

# In kích thước của tập dữ liệu (số dòng, số cột)
print(f"Kích thước tập dữ liệu ban đầu: {diamonds.shape}\n")

# Hiển thị thông tin tổng quan
diamonds.info()

# In tiêu đề thông báo cho phần hiển thị 5 dòng đầu
print("\n--- 5 dòng đầu tiên ---")
# Hiển thị 5 dòng dữ liệu đầu tiên
print(diamonds.head())

# In tiêu đề thông báo cho phần kiểm tra các giá trị phân loại
print("\n--- Giá trị phân loại của các thuộc tính dạng chuỗi ---")
# Lấy và in danh sách các giá trị duy nhất (không trùng lặp) của cột 'cut' (vết cắt)
print(f"Cut: {diamonds['cut'].unique()}")
# Lấy và in danh sách các giá trị duy nhất của cột 'color' (màu sắc)
print(f"Color: {diamonds['color'].unique()}")
# Lấy và in danh sách các giá trị duy nhất của cột 'clarity' (độ tinh khiết)
print(f"Clarity: {diamonds['clarity'].unique()}")

# In tiêu đề thông báo cho phần thống kê mô tả
print("\n--- Thống kê mô tả ---")
# Tính toán các chỉ số thống kê cơ bản cho các cột số
print(diamonds.describe())

Kích thước tập dữ liệu ban đầu: (53943, 11)

<class 'pandas.DataFrame'>
RangeIndex: 53943 entries, 0 to 53942
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  53943 non-null  int64  
 1   carat       53943 non-null  float64
 2   cut         53943 non-null  str    
 3   color       53943 non-null  str    
 4   clarity     53943 non-null  str    
 5   depth       53943 non-null  float64
 6   table       53943 non-null  float64
 7   price       53943 non-null  int64  
 8   x           53943 non-null  float64
 9   y           53943 non-null  float64
 10  z           53943 non-null  float64
dtypes: float64(6), int64(2), str(3)
memory usage: 4.5 MB

--- 5 dòng đầu tiên ---
   Unnamed: 0  carat      cut color clarity  depth  table  price     x     y  \
0           1   0.23    Ideal     E     SI2   61.5   55.0    326  3.95  3.98   
1           2   0.21  Premium     E     SI1   59.8   61.0    326  3.89  3.84   
2

## Tiền xử lý dữ liệu

In [2]:
# Nạp công cụ OrdinalEncoder từ thư viện scikit-learn để mã hóa các biến phân loại có thứ tự
from sklearn.preprocessing import OrdinalEncoder

# 1: Loại bỏ thuộc tính không mang thông tin
# Kiểm tra xem cột số thứ tự thừa 'Unnamed: 0' có tồn tại trong DataFrame hay không
if 'Unnamed: 0' in diamonds.columns:
    # Nếu có, xóa bỏ cột này khỏi tập dữ liệu
    diamonds = diamonds.drop(columns=['Unnamed: 0'])

# 2: Loại bỏ dữ liệu nhiễu (kích thước x, y, z <= 0)
# Lấy tổng số dòng dữ liệu ban đầu trước khi lọc
initial_rows = len(diamonds)

# Lọc giữ lại các dòng có cả 3 kích thước (x, y, z) đều lớn hơn 0 và tạo bản sao dữ liệu mới
diamonds = diamonds[(diamonds['x'] > 0) & (diamonds['y'] > 0) & (diamonds['z'] > 0)].copy()

# Tính số lượng dòng dữ liệu bị loại bỏ (bằng số dòng ban đầu trừ số dòng sau khi lọc)
invalid_dimensions = initial_rows - len(diamonds)

# In số lượng dòng nhiễu đã bị loại bỏ ra màn hình (thêm \n ở đầu để cách dòng trên)
print(f"\nĐã loại bỏ {invalid_dimensions} dòng dữ liệu nhiễu (x, y, z <= 0).")

# 3: Loại bỏ các dòng trùng lặp hoàn toàn
# Đếm tổng số dòng dữ liệu bị lặp lại hoàn toàn trong tập dữ liệu
num_duplicates = diamonds.duplicated().sum()

# Xóa bỏ các dòng trùng lặp, chỉ giữ lại 1 bản ghi đại diện
diamonds = diamonds.drop_duplicates()

# In số lượng dòng trùng lặp đã xóa ra màn hình (thêm \n ở đầu để cách với dòng trên)
print(f"\nĐã loại bỏ {num_duplicates} dòng trùng lặp.")

# 4: Mã hóa thứ tự (Ordinal Encoding) cho các thuộc tính phân loại
# Định nghĩa danh sách các mức độ chất lượng của vết cắt 'cut' theo thứ tự tăng dần (Fair là thấp nhất, Ideal là cao nhất)
cut_categories = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']

# Định nghĩa danh sách màu sắc 'color' theo thứ tự tăng dần từ kém nhất đến tốt nhất (J -> D)
color_categories = ['J', 'I', 'H', 'G', 'F', 'E', 'D']

# Định nghĩa danh sách độ tinh khiết 'clarity' theo thứ tự tăng dần từ kém nhất đến tốt nhất (I1 -> IF)
clarity_categories = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']

# Khởi tạo đối tượng OrdinalEncoder với danh sách thứ tự đã thiết lập cho từng cột
encoder = OrdinalEncoder(categories=[cut_categories, color_categories, clarity_categories])

# chuyển đổi các cột dữ liệu dạng chữ (phân loại) thành dạng số dựa trên thứ tự đã định nghĩa
diamonds[['cut', 'color', 'clarity']] = encoder.fit_transform(diamonds[['cut', 'color', 'clarity']])

# 5: Kiểm tra dữ liệu sau tiền xử lý
# In kích thước mới của tập dữ liệu (số dòng, số cột) sau khi đã làm sạch và chuyển đổi
print(f"\nKích thước tập dữ liệu sau tiền xử lý: {diamonds.shape}\n")

# In tiêu đề thông báo hiển thị 5 dòng đầu
print("--- 5 dòng dữ liệu đầu sau tiền xử lý ---")

# Hiển thị 5 dòng dữ liệu đầu tiên sau tiền xử lý
print(diamonds.head())


Đã loại bỏ 20 dòng dữ liệu nhiễu (x, y, z <= 0).

Đã loại bỏ 148 dòng trùng lặp.

Kích thước tập dữ liệu sau tiền xử lý: (53775, 10)

--- 5 dòng dữ liệu đầu sau tiền xử lý ---
   carat  cut  color  clarity  depth  table  price     x     y     z
0   0.23  4.0    5.0      1.0   61.5   55.0    326  3.95  3.98  2.43
1   0.21  3.0    5.0      2.0   59.8   61.0    326  3.89  3.84  2.31
2   0.23  1.0    5.0      4.0   56.9   65.0    327  4.05  4.07  2.31
3   0.29  3.0    1.0      3.0   62.4   58.0    334  4.20  4.23  2.63
4   0.31  1.0    0.0      1.0   63.3   58.0    335  4.34  4.35  2.75


## Chia tập dữ liệu

In [3]:
from sklearn.model_selection import KFold

# X là dữ liệu các thuộc tính
X = diamonds.drop(columns=["price"])
# y là dữ liệu nhãn
y = diamonds["price"]

# Chia dữ liệu thành 5 fold
Kf = KFold(n_splits = 5, shuffle = True, random_state=42)

## Traning và kết quả đánh giá

In [4]:
# Import mô hình cây quyết định hồi quy
from sklearn.tree import DecisionTreeRegressor
# Import các hàm đánh giá
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# Import K-Fold
from sklearn.model_selection import KFold
# Import thư viện đo thời gian
import time
# Import thư viện numpy
import numpy as np
# Import thư viện pandas
import pandas as pd

# Tạo mảng lưu trữ kết quả đánh giá cho từng Fold
results = []

# Bắt đầu đo thời gian huấn luyện
start_time = time.perf_counter()
# Duyệt qua từng Fold
for fold, (train_index, test_index) in enumerate(Kf.split(X), start=1):

    # Chia dữ liệu Train và Test
    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]

    y_train = y.iloc[train_index]
    y_test = y.iloc[test_index]

    # Khởi tạo mô hình
    model = DecisionTreeRegressor(
        criterion="squared_error",
        random_state=42
    )

    # Bắt đầu đo thời gian huấn luyện
    start_train_time = time.perf_counter()

    # Huấn luyện mô hình trên tập Train
    model.fit(X_train, y_train)

    # Tính thời gian huấn luyện
    training_time = time.perf_counter() - start_train_time

    # Dự đoán trên tập Train
    y_pred_train = model.predict(X_train)
    # Dự đoán trên tập Test
    y_pred_test = model.predict(X_test)

    # Tính các chỉ số Train
    mae_train = mean_absolute_error(y_train, y_pred_train)
    mse_train = mean_squared_error(y_train, y_pred_train)
    rmse_train = np.sqrt(mse_train)
    r2_train = r2_score(y_train, y_pred_train)

    # Tính các chỉ số Test
    mae_test = mean_absolute_error(y_test, y_pred_test)
    mse_test = mean_squared_error(y_test, y_pred_test)
    rmse_test = np.sqrt(mse_test)
    r2_test = r2_score(y_test, y_pred_test)

    # Lưu kết quả
    results.append({
        "Fold": fold,
        "MAE Train": mae_train,
        "MSE Train": mse_train,
        "RMSE Train": rmse_train,
        "R2 Train": r2_train,
        "MAE Test": mae_test,
        "MSE Test": mse_test,
        "RMSE Test": rmse_test,
        "R2 Test": r2_test,
        "Training Time (s)": training_time
    })

# Bắt đầu đo thời gian huấn luyện
train_time = time.perf_counter() - start_time

# Tạo bảng kết quả từng Fold
results_df = pd.DataFrame(results)

# Tính giá trị trung bình của 5 Fold
mean_result = results_df.drop(columns=["Fold"]).mean()

# Tạo dòng trung bình
mean_row = pd.DataFrame([mean_result])
mean_row.insert(0, "Fold", "Average")

# Ghép kết quả từng Fold và trung bình
final_results = pd.concat(
    [results_df, mean_row],
    ignore_index=True
)

# In bảng kết quả
print("KẾT QUẢ DECISION TREE - K-FOLD CROSS VALIDATION")
print(final_results.to_string(index=False, float_format="%.4f"))
print(f"Tổng thời gian train: {train_time:.4f} giây")

KẾT QUẢ DECISION TREE - K-FOLD CROSS VALIDATION
   Fold  MAE Train  MSE Train  RMSE Train  R2 Train  MAE Test    MSE Test  RMSE Test  R2 Test  Training Time (s)
      1     0.4436    78.5161      8.8609    1.0000  353.9126 517060.7204   719.0693   0.9673             0.1851
      2     0.4847    76.0481      8.7206    1.0000  358.3698 537332.8580   733.0299   0.9655             0.1792
      3     0.4219    51.6537      7.1871    1.0000  346.7435 515098.5916   717.7037   0.9675             0.1787
      4     0.3610    54.3832      7.3745    1.0000  359.4477 536438.4644   732.4196   0.9670             0.1788
      5     0.4204    49.7185      7.0511    1.0000  359.8536 563828.0803   750.8849   0.9647             0.1767
Average     0.4263    62.0639      7.8388    1.0000  355.6655 533951.7429   730.6215   0.9664             0.1797
Tổng thời gian train: 1.0075 giây
